In [ ]:
"""
SHWD V1 Benchmark Camera-Ready Controlled Ablation Suite (Seed 2026)
===========================================================
Account: shwd_v1_archive
Protocol:
  - Dataset: SHWD (VOC2028, 7,581 images)
      * train: ImageSets/Main/train.txt (5,457 images)
      * val:   ImageSets/Main/val.txt   (607 images, used for best.pt selection)
      * test:  ImageSets/Main/test.txt  (1,517 images, STRICTLY HELD-OUT)
      * classes: 0='hat' (helmeted head), 1='person' (unhelmeted head)
  - External Zero-Shot Benchmark: Hard Hat Workers (HHW, 7,063 images; Test=1,766)
      * 2-Class Head-Level: helmet->0 (hat), head->1 (person) [sparse full-body person ignored]
      * Hat-Only: helmet->0 (hat)
      * imgsz=640, half=False, augment=False (no TTA)
  - Warm-Start Weight Transfer: All architectural insertions (CoordConv, Neck-Only RepConv,
    Residual BiFormer at P4/P5) preserve 100% of pretrained COCO representations at Epoch 0.
  - Strictly Verified DDP Subprocess Injection & Deploy Fusion.
"""

import os
import sys
import gc
import json
import time
import copy
import site
import shutil
import zipfile
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

print("=" * 80, flush=True)
print("[SHWD V1 Benchmark] CONTROLLED ABLATION SUITE (SEED 2026)", flush=True)
print("=" * 80, flush=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3.0", "scipy", "pandas", "onnx", "onnxruntime-gpu"],
    check=False,
)

os.environ["TENSORBOARD_BINARY"] = ""
os.environ["YOLO_VERBOSE"] = "False"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO, settings
settings.update({"tensorboard": False})

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    ngpu = torch.cuda.device_count()
    for i in range(ngpu):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU[{i}]: {p.name} ({p.total_memory / (1024**3):.2f} GB)", flush=True)
    TRAIN_DEVICE = "0,1" if ngpu >= 2 else "0"
    EVAL_DEVICE = "0"
else:
    TRAIN_DEVICE = "cpu"
    EVAL_DEVICE = "cpu"

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

# ==============================================================================
# 1. WRITE SELF-CONTAINED RESEARCH MODULES & DDP TRAINER TO WORKING + SITE-PKGS
# ==============================================================================
CUSTOM_MODULES_CODE = '''
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F

def autopad(k: int, p: int | None = None, d: int = 1) -> int:
    if p is None:
        p = d * (k - 1) // 2
    return p

class ConvBNAct(nn.Module):
    def __init__(self, c1: int, c2: int, k: int = 1, s: int = 1, p: int | None = None, g: int = 1, d: int = 1, act: bool = True) -> None:
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))

class CoordConv(nn.Module):
    def __init__(self, c1: int, c2: int, k: int = 3, s: int = 1, with_r: bool = False) -> None:
        super().__init__()
        self.with_r = with_r
        extra = 3 if with_r else 2
        self.conv = ConvBNAct(c1 + extra, c2, k=k, s=s)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        yy = torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype).view(1, 1, h, 1).expand(b, 1, h, w)
        xx = torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype).view(1, 1, 1, w).expand(b, 1, h, w)
        coords = [xx, yy]
        if self.with_r:
            rr = torch.sqrt(torch.clamp(xx.square() + yy.square(), min=0.0))
            coords.append(rr)
        return self.conv(torch.cat([x, *coords], dim=1))

    def fuse(self) -> None: pass
    def fuse_convs(self) -> None: pass
    def forward_fuse(self, x: torch.Tensor) -> torch.Tensor: return self.forward(x)

class RepConv(nn.Module):
    def __init__(self, c1: int, c2: int, k: int = 3, s: int = 1, deploy: bool = False, act: bool = True) -> None:
        super().__init__()
        assert k == 3
        self.deploy = deploy
        self.in_channels = c1
        self.out_channels = c2
        self.stride = s
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()
        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, 3, s, 1, bias=True)
        else:
            self.rbr_dense = nn.Sequential(nn.Conv2d(c1, c2, 3, s, 1, bias=False), nn.BatchNorm2d(c2))
            self.rbr_1x1 = nn.Sequential(nn.Conv2d(c1, c2, 1, s, 0, bias=False), nn.BatchNorm2d(c2))
            self.rbr_identity = nn.BatchNorm2d(c1) if c1 == c2 and s == 1 else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        out = self.rbr_dense(x) + self.rbr_1x1(x)
        if self.rbr_identity is not None:
            out = out + self.rbr_identity(x)
        return self.act(out)

    @staticmethod
    def _fuse_conv_bn(branch, channels: int):
        if isinstance(branch, nn.Sequential):
            conv, bn = branch[0], branch[1]
            kernel = conv.weight
        else:
            bn = branch
            kernel = torch.zeros((channels, channels, 3, 3), device=bn.weight.device, dtype=bn.weight.dtype)
            for i in range(channels):
                kernel[i, i, 1, 1] = 1.0
        std = torch.sqrt(bn.running_var + bn.eps)
        t = (bn.weight / std).reshape(-1, 1, 1, 1)
        return kernel * t, bn.bias - bn.running_mean * bn.weight / std

    @staticmethod
    def _pad_1x1_to_3x3(kernel: torch.Tensor) -> torch.Tensor:
        return kernel if kernel.size(2) == 3 else F.pad(kernel, [1, 1, 1, 1])

    def get_equivalent_kernel_bias(self):
        k3, b3 = self._fuse_conv_bn(self.rbr_dense, self.out_channels)
        k1, b1 = self._fuse_conv_bn(self.rbr_1x1, self.out_channels)
        if self.rbr_identity is None:
            kid, bid = torch.zeros_like(k3), torch.zeros_like(b3)
        else:
            kid, bid = self._fuse_conv_bn(self.rbr_identity, self.out_channels)
        return k3 + self._pad_1x1_to_3x3(k1) + kid.to(k3.device), b3 + b1 + bid.to(b3.device)

    def switch_to_deploy(self) -> None:
        if self.deploy:
            return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(self.in_channels, self.out_channels, 3, self.stride, 1, bias=True).to(
            device=kernel.device, dtype=kernel.dtype
        )
        self.rbr_reparam.weight.data.copy_(kernel.detach())
        self.rbr_reparam.bias.data.copy_(bias.detach())
        self.conv = self.rbr_reparam
        del self.rbr_dense
        del self.rbr_1x1
        if hasattr(self, "rbr_identity"):
            del self.rbr_identity
        self.deploy = True

    def fuse_convs(self) -> None: self.switch_to_deploy()
    def fuse(self) -> None: self.switch_to_deploy()
    def forward_fuse(self, x: torch.Tensor) -> torch.Tensor:
        if hasattr(self, "rbr_reparam"):
            return self.act(self.rbr_reparam(x))
        return self.forward(x)

class BiFormerBlockLite(nn.Module):
    def __init__(self, channels: int, num_heads: int = 4, region_size: int = 8, topk: int = 4) -> None:
        super().__init__()
        assert channels % num_heads == 0
        self.channels = channels
        self.num_heads = num_heads
        self.region_size = region_size
        self.topk = topk
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.proj = nn.Conv2d(channels, channels, 1, bias=False)
        self.norm = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        rs = self.region_size
        pad_h = (rs - h % rs) % rs
        pad_w = (rs - w % rs) % rs
        x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        hp, wp = x_pad.shape[-2:]
        gh, gw = hp // rs, wp // rs
        R = gh * gw
        T = rs * rs

        q, k, v = self.qkv(x_pad).chunk(3, dim=1)
        q_tokens = q.view(b, c, gh, rs, gw, rs).permute(0, 2, 4, 3, 5, 1).reshape(b, R, T, c)
        k_tokens = k.view(b, c, gh, rs, gw, rs).permute(0, 2, 4, 3, 5, 1).reshape(b, R, T, c)
        v_tokens = v.view(b, c, gh, rs, gw, rs).permute(0, 2, 4, 3, 5, 1).reshape(b, R, T, c)

        q_reg = q_tokens.mean(dim=2).float()
        k_reg = k_tokens.mean(dim=2).float()
        route_logits = torch.matmul(q_reg, k_reg.transpose(-1, -2)) / (c ** 0.5)
        route_logits = torch.nan_to_num(route_logits, nan=-1e4, posinf=1e4, neginf=-1e4)
        topk = min(self.topk, R)
        route_idx = route_logits.topk(topk, dim=-1).indices

        idx_exp = route_idx.unsqueeze(-1).unsqueeze(-1).expand(b, R, topk, T, c)
        k_all = k_tokens.unsqueeze(1).expand(b, R, R, T, c)
        v_all = v_tokens.unsqueeze(1).expand(b, R, R, T, c)
        k_sel = torch.gather(k_all, 2, idx_exp).reshape(b, R, topk * T, c)
        v_sel = torch.gather(v_all, 2, idx_exp).reshape(b, R, topk * T, c)

        head_dim = c // self.num_heads
        qh = q_tokens.view(b, R, T, self.num_heads, head_dim).permute(0, 1, 3, 2, 4)
        kh = k_sel.view(b, R, topk * T, self.num_heads, head_dim).permute(0, 1, 3, 2, 4)
        vh = v_sel.view(b, R, topk * T, self.num_heads, head_dim).permute(0, 1, 3, 2, 4)

        attn_scores = torch.matmul(qh.float(), kh.float().transpose(-1, -2)) / (head_dim ** 0.5)
        attn_scores = torch.clamp(torch.nan_to_num(attn_scores, nan=-50.0, posinf=50.0, neginf=-50.0), -50.0, 50.0)
        attn = torch.softmax(attn_scores, dim=-1)
        out = torch.matmul(attn, vh.float()).to(qh.dtype).permute(0, 1, 3, 2, 4).reshape(b, R, T, c)

        y = out.view(b, gh, gw, rs, rs, c).permute(0, 5, 1, 3, 2, 4).reshape(b, c, hp, wp)
        y = y[:, :, :h, :w]
        return torch.nan_to_num(x + self.norm(self.proj(torch.nan_to_num(y, nan=0.0))), nan=0.0)

    def fuse(self) -> None: pass
    def fuse_convs(self) -> None: pass
    def forward_fuse(self, x: torch.Tensor) -> torch.Tensor: return self.forward(x)

class BiFormerSequentialWrapper(nn.Module):
    def __init__(self, orig_layer: nn.Module, biformer: nn.Module) -> None:
        super().__init__()
        self.orig_layer = orig_layer
        self.biformer = biformer
        self.i = getattr(orig_layer, "i", -1)
        self.f = getattr(orig_layer, "f", -1)
        self.type = "BiFormerSequentialWrapper"
        self.scale = nn.Parameter(torch.full((1,), 1e-4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig_out = self.orig_layer(x)
        return orig_out + self.scale * self.biformer(orig_out)

    def fuse(self) -> None:
        if hasattr(self.orig_layer, "fuse"):
            self.orig_layer.fuse()

    def fuse_convs(self) -> None:
        if hasattr(self.orig_layer, "fuse_convs"):
            self.orig_layer.fuse_convs()
'''

ABLATION_TRAINER_CODE = '''
import os
import copy
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import ultralytics.nn.modules as un_mod
import ultralytics.nn.tasks as un_tasks
import ultralytics.utils.loss as ul_loss

from custom_ablation_modules import ConvBNAct, CoordConv, RepConv, BiFormerBlockLite, BiFormerSequentialWrapper

for tgt in (un_mod, un_tasks):
    setattr(tgt, "ConvBNAct", ConvBNAct)
    setattr(tgt, "CoordConv", CoordConv)
    setattr(tgt, "RepConv", RepConv)
    setattr(tgt, "BiFormerBlockLite", BiFormerBlockLite)
    setattr(tgt, "BiFormerSequentialWrapper", BiFormerSequentialWrapper)

def _get_out_channels(layer: nn.Module) -> int:
    if hasattr(layer, "conv") and hasattr(layer.conv, "out_channels"):
        return layer.conv.out_channels
    if hasattr(layer, "cv2") and hasattr(layer.cv2, "conv"):
        return layer.cv2.conv.out_channels
    raise ValueError(f"Cannot determine output channels for {type(layer)}")

def _wrap_repconv_from_conv(conv_mod: nn.Module) -> RepConv:
    c1, c2 = conv_mod.conv.in_channels, conv_mod.conv.out_channels
    s = conv_mod.conv.stride[0]
    rc = RepConv(c1, c2, k=3, s=s, deploy=False)
    rc.rbr_dense[1].eps = conv_mod.bn.eps
    rc.rbr_1x1[1].eps = conv_mod.bn.eps
    with torch.no_grad():
        rc.rbr_dense[0].weight.copy_(conv_mod.conv.weight)
        rc.rbr_dense[1].weight.copy_(conv_mod.bn.weight)
        rc.rbr_dense[1].bias.copy_(conv_mod.bn.bias)
        rc.rbr_dense[1].running_mean.copy_(conv_mod.bn.running_mean)
        rc.rbr_dense[1].running_var.copy_(conv_mod.bn.running_var)
        rc.rbr_1x1[1].weight.zero_()
        rc.rbr_1x1[1].bias.zero_()
        if rc.rbr_identity is not None:
            rc.rbr_identity.eps = conv_mod.bn.eps
            rc.rbr_identity.weight.zero_()
            rc.rbr_identity.bias.zero_()
    rc.i = getattr(conv_mod, "i", -1)
    rc.f = getattr(conv_mod, "f", -1)
    rc.type = "RepConv"
    return rc

def build_ablation_model(ab_id: str, base_nc2_weight: str):
    yolo = YOLO(base_nc2_weight)
    m = yolo.model

    if ab_id in ("A1", "A2", "A3", "A4"):
        orig0 = m.model[0]
        c1, c2 = orig0.conv.in_channels, orig0.conv.out_channels
        cc = CoordConv(c1, c2, k=3, s=2, with_r=False)
        cc.conv.bn.eps = orig0.bn.eps
        with torch.no_grad():
            cc.conv.conv.weight.zero_()
            cc.conv.conv.weight[:, :c1].copy_(orig0.conv.weight)
            cc.conv.bn.weight.copy_(orig0.bn.weight)
            cc.conv.bn.bias.copy_(orig0.bn.bias)
            cc.conv.bn.running_mean.copy_(orig0.bn.running_mean)
            cc.conv.bn.running_var.copy_(orig0.bn.running_var)
        cc.i, cc.f, cc.type = orig0.i, orig0.f, "CoordConv"
        m.model[0] = cc

    if ab_id in ("A2", "A3", "A4"):
        for idx in range(11, 23):
            layer = m.model[idx]
            if hasattr(layer, "conv") and hasattr(layer.conv, "kernel_size") and layer.conv.kernel_size == (3, 3):
                m.model[idx] = _wrap_repconv_from_conv(layer)
            for name, sub in list(layer.named_modules()):
                if type(sub).__name__ == "Conv" and hasattr(sub, "conv") and sub.conv.kernel_size == (3, 3) and sub.conv.groups == 1:
                    rc = _wrap_repconv_from_conv(sub)
                    parts = name.split(".")
                    parent = layer
                    for p in parts[:-1]:
                        parent = getattr(parent, p)
                    setattr(parent, parts[-1], rc)

    if ab_id in ("A3", "A4"):
        for idx in (19, 22):
            layer = m.model[idx]
            c_out = _get_out_channels(layer)
            bf = BiFormerBlockLite(channels=c_out, num_heads=4, region_size=8, topk=4)
            wrap = BiFormerSequentialWrapper(layer, bf)
            m.model[idx] = wrap

    yolo.model = m
    yolo.ckpt = {"model": yolo.model}
    return yolo

class AblationBboxLoss(ul_loss.BboxLoss):
    def forward(self, pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask, *args, **kwargs):
        cur_ab = os.environ.get("CURRENT_ABLATION_ID", "A0")
        if cur_ab != "A4":
            return super().forward(pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask, *args, **kwargs)

        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
        p_box = pred_bboxes[fg_mask]
        t_box = target_bboxes[fg_mask]
        if p_box.shape[0] > 0:
            px1, py1, px2, py2 = p_box.unbind(-1)
            tx1, ty1, tx2, ty2 = t_box.unbind(-1)
            pw = (px2 - px1).clamp(min=1e-7)
            ph = (py2 - py1).clamp(min=1e-7)
            tw = (tx2 - tx1).clamp(min=1e-7)
            th = (ty2 - ty1).clamp(min=1e-7)
            inter_x1 = torch.maximum(px1, tx1)
            inter_y1 = torch.maximum(py1, ty1)
            inter_x2 = torch.minimum(px2, tx2)
            inter_y2 = torch.minimum(py2, ty2)
            inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
            union = pw * ph + tw * th - inter + 1e-7
            iou = (inter / union).clamp(min=1e-7, max=1.0)
            pcx, pcy = (px1 + px2) / 2.0, (py1 + py2) / 2.0
            tcx, tcy = (tx1 + tx2) / 2.0, (ty1 + ty2) / 2.0
            center_dist = (pcx - tcx).square() + (pcy - tcy).square()
            cw = (torch.maximum(px2, tx2) - torch.minimum(px1, tx1)).clamp(min=1e-7)
            ch = (torch.maximum(py2, ty2) - torch.minimum(py1, ty1)).clamp(min=1e-7)
            c2 = cw.square() + ch.square() + 1e-7
            eiou = 1.0 - iou + center_dist / c2 + (pw - tw).square() / (cw.square() + 1e-7) + (ph - th).square() / (ch.square() + 1e-7)
            loss_box_sample = iou.pow(0.5) * eiou
            loss_iou = (loss_box_sample.unsqueeze(-1) * weight).sum() / target_scores_sum
        else:
            loss_iou = torch.tensor(0.0, device=pred_dist.device)

        if self.dfl_loss and p_box.shape[0] > 0:
            try:
                target_ltrb = ul_loss.bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1)
            except AttributeError:
                from ultralytics.utils.tal import bbox2dist
                target_ltrb = bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1)
            loss_dfl = self.dfl_loss(pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max), target_ltrb[fg_mask]) * weight
            loss_dfl = loss_dfl.sum() / target_scores_sum
        else:
            loss_dfl = torch.tensor(0.0, device=pred_dist.device)
        return loss_iou, loss_dfl

ul_loss.BboxLoss = AblationBboxLoss

class ControlledAblationTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        ab_id = os.environ.get("CURRENT_ABLATION_ID", "A0")
        bw = os.environ.get("CURRENT_BASE_WEIGHT", "/kaggle/working/yolo11s_shwd_nc2_init.pt")
        built = build_ablation_model(ab_id, bw)
        return copy.deepcopy(built.model).float()

    def _setup_train(self, *args, **kwargs):
        super()._setup_train(*args, **kwargs)
        ab_id = os.environ.get("CURRENT_ABLATION_ID", "A0")
        unwrapped = self.model.module if hasattr(self.model, "module") else self.model
        n_coord = sum(1 for mod in unwrapped.modules() if isinstance(mod, CoordConv))
        n_rep = sum(1 for mod in unwrapped.modules() if isinstance(mod, RepConv))
        n_bf = sum(1 for mod in unwrapped.modules() if isinstance(mod, BiFormerSequentialWrapper))
        if ab_id in ("A1", "A2", "A3", "A4"):
            assert n_coord == 1, f"Expected 1 CoordConv for {ab_id}, got {n_coord}"
        if ab_id in ("A2", "A3", "A4"):
            assert n_rep == 12, f"Expected 12 Neck RepConv for {ab_id}, got {n_rep}"
        if ab_id in ("A3", "A4"):
            assert n_bf == 2, f"Expected 2 BiFormer wrappers for {ab_id}, got {n_bf}"
        if ab_id == "A4":
            crit = unwrapped.init_criterion()
            assert type(crit.bbox_loss).__name__ == "AblationBboxLoss", f"AblationBboxLoss not active! Got {type(crit.bbox_loss)}"
        print(f"[DDP VERIFIED {ab_id}] CoordConv={n_coord}, RepConv={n_rep}, BiFormer={n_bf}, BboxLossModule={ul_loss.BboxLoss.__name__}", flush=True)
'''

(WORK_DIR / "custom_ablation_modules.py").write_text(CUSTOM_MODULES_CODE.strip() + "\n", encoding="utf-8")
(WORK_DIR / "ablation_trainer.py").write_text(ABLATION_TRAINER_CODE.strip() + "\n", encoding="utf-8")

for sp in site.getsitepackages():
    try:
        (Path(sp) / "custom_ablation_modules.py").write_text(CUSTOM_MODULES_CODE.strip() + "\n", encoding="utf-8")
        (Path(sp) / "ablation_trainer.py").write_text(ABLATION_TRAINER_CODE.strip() + "\n", encoding="utf-8")
    except Exception:
        pass

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.environ["PYTHONPATH"] = f"{WORK_DIR}:{os.environ.get('PYTHONPATH', '')}".rstrip(":")

# Patch physical ultralytics/utils/loss.py for DDP worker subprocesses
import ultralytics.utils.loss as ul_loss
loss_file = Path(ul_loss.__file__).resolve()
loss_src = loss_file.read_text(encoding="utf-8")
marker = "# === SHWD V1 Benchmark ABLATION LOSS PATCH ==="
base_loss_src = loss_src.split(marker)[0].rstrip() if marker in loss_src else loss_src.rstrip()

LOSS_PATCH = '''
# === SHWD V1 Benchmark ABLATION LOSS PATCH ===
import os as _os_patch
import torch as _torch_patch
class AblationBboxLoss(BboxLoss):
    def forward(self, pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask, *args, **kwargs):
        cur_ab = _os_patch.environ.get("CURRENT_ABLATION_ID", "A0")
        if cur_ab != "A4":
            return super().forward(pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask, *args, **kwargs)
        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
        p_box = pred_bboxes[fg_mask]
        t_box = target_bboxes[fg_mask]
        if p_box.shape[0] > 0:
            px1, py1, px2, py2 = p_box.unbind(-1)
            tx1, ty1, tx2, ty2 = t_box.unbind(-1)
            pw = (px2 - px1).clamp(min=1e-7)
            ph = (py2 - py1).clamp(min=1e-7)
            tw = (tx2 - tx1).clamp(min=1e-7)
            th = (ty2 - ty1).clamp(min=1e-7)
            inter_x1 = _torch_patch.maximum(px1, tx1)
            inter_y1 = _torch_patch.maximum(py1, ty1)
            inter_x2 = _torch_patch.minimum(px2, tx2)
            inter_y2 = _torch_patch.minimum(py2, ty2)
            inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
            union = pw * ph + tw * th - inter + 1e-7
            iou = (inter / union).clamp(min=1e-7, max=1.0)
            pcx, pcy = (px1 + px2) / 2.0, (py1 + py2) / 2.0
            tcx, tcy = (tx1 + tx2) / 2.0, (ty1 + ty2) / 2.0
            center_dist = (pcx - tcx).square() + (pcy - tcy).square()
            cw = (_torch_patch.maximum(px2, tx2) - _torch_patch.minimum(px1, tx1)).clamp(min=1e-7)
            ch = (_torch_patch.maximum(py2, ty2) - _torch_patch.minimum(py1, ty1)).clamp(min=1e-7)
            c2 = cw.square() + ch.square() + 1e-7
            eiou = 1.0 - iou + center_dist / c2 + (pw - tw).square() / (cw.square() + 1e-7) + (ph - th).square() / (ch.square() + 1e-7)
            loss_box_sample = iou.pow(0.5) * eiou
            loss_iou = (loss_box_sample.unsqueeze(-1) * weight).sum() / target_scores_sum
        else:
            loss_iou = _torch_patch.tensor(0.0, device=pred_dist.device)
        if self.dfl_loss and p_box.shape[0] > 0:
            try:
                target_ltrb = bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1)
            except NameError:
                from ultralytics.utils.tal import bbox2dist as _b2d
                target_ltrb = _b2d(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1)
            loss_dfl = self.dfl_loss(pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max), target_ltrb[fg_mask]) * weight
            loss_dfl = loss_dfl.sum() / target_scores_sum
        else:
            loss_dfl = _torch_patch.tensor(0.0, device=pred_dist.device)
        return loss_iou, loss_dfl
BboxLoss = AblationBboxLoss
'''
loss_file.write_text(base_loss_src + "\n" + LOSS_PATCH, encoding="utf-8")
print("[OK] Patched ultralytics/utils/loss.py for DDP subprocesses.", flush=True)

from custom_ablation_modules import CoordConv, RepConv, BiFormerBlockLite, BiFormerSequentialWrapper
from ablation_trainer import ControlledAblationTrainer, build_ablation_model

# ==============================================================================
# 2. PREPARE SHWD (VOC2028) OFFICIAL TRAIN / VAL / TEST + HHW ZERO-SHOT
# ==============================================================================
voc_root = None
for p in Path("/kaggle/input").rglob("JPEGImages"):
    if (p.parent / "Annotations").is_dir() and (p.parent / "ImageSets" / "Main" / "test.txt").exists():
        voc_root = p.parent
        break
assert voc_root is not None, "VOC2028 dataset not found in /kaggle/input!"
print(f"[DATASET] Found SHWD/VOC2028 at: {voc_root}", flush=True)

SHWD_DIR = WORK_DIR / "SHWD_YOLO"
for sp in ("train", "val", "test"):
    (SHWD_DIR / "images" / sp).mkdir(parents=True, exist_ok=True)
    (SHWD_DIR / "labels" / sp).mkdir(parents=True, exist_ok=True)

SHWD_MAP = {"hat": 0, "helmet": 0, "person": 1, "head": 1}

def build_voc_yolo(root_dir: Path, id_file: Path, out_dir: Path, split_name: str, cls_map: dict):
    ids = [x.strip() for x in id_file.read_text(encoding="utf-8").splitlines() if x.strip()]
    n_ok, counts = 0, {v: 0 for v in set(cls_map.values())}
    for stem in ids:
        xp = root_dir / "Annotations" / f"{stem}.xml"
        if not xp.exists():
            continue
        img_p = None
        for ext in (".jpg", ".JPG", ".png", ".jpeg"):
            cand = root_dir / "JPEGImages" / f"{stem}{ext}"
            if cand.exists():
                img_p = cand
                break
        if img_p is None:
            continue
        root = ET.parse(xp).getroot()
        sz = root.find("size")
        iw, ih = int(sz.find("width").text), int(sz.find("height").text)
        if iw <= 0 or ih <= 0:
            continue
        lines = []
        for obj in root.findall("object"):
            cname = obj.find("name").text.strip().lower()
            if cname not in cls_map:
                continue
            cid = cls_map[cname]
            bb = obj.find("bndbox")
            xmin = max(0.0, min(float(bb.find("xmin").text), float(iw)))
            ymin = max(0.0, min(float(bb.find("ymin").text), float(ih)))
            xmax = max(0.0, min(float(bb.find("xmax").text), float(iw)))
            ymax = max(0.0, min(float(bb.find("ymax").text), float(ih)))
            if xmax <= xmin or ymax <= ymin:
                continue
            cx = ((xmin + xmax) / 2.0) / iw
            cy = ((ymin + ymax) / 2.0) / ih
            bw = (xmax - xmin) / iw
            bh = (ymax - ymin) / ih
            lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            counts[cid] += 1
        dst_img = out_dir / "images" / split_name / img_p.name
        if not dst_img.exists():
            try:
                os.symlink(img_p, dst_img)
            except Exception:
                shutil.copy2(img_p, dst_img)
        (out_dir / "labels" / split_name / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        n_ok += 1
    print(f"  SHWD {split_name}: {n_ok} images, boxes={counts}", flush=True)

for sp in ("train", "val", "test"):
    build_voc_yolo(voc_root, voc_root / "ImageSets" / "Main" / f"{sp}.txt", SHWD_DIR, sp, SHWD_MAP)

shwd_yaml = SHWD_DIR / "shwd.yaml"
shwd_yaml.write_text(
    f"path: {SHWD_DIR.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 2\nnames: ['hat', 'person']\n",
    encoding="utf-8",
)

# Prepare HHW Zero-Shot Benchmark (Test=1,766 and Full=7,063) under 2-Class and Hat-Only
hhw_root = None
for p in Path("/kaggle/input").rglob("Annotation"):
    if (p.parent / "JPEGImage").is_dir() and p.parent.name in ("Test", "Train"):
        hhw_root = p.parent.parent
        break

hhw_configs = {}
if hhw_root is not None:
    print(f"[DATASET] Found HHW at: {hhw_root}", flush=True)
    for mode, cmap, names_list in [
        ("hhw_2class", {"helmet": 0, "head": 1}, ["hat", "person"]),
        ("hhw_hatonly", {"helmet": 0}, ["hat"]),
    ]:
        for subset_name, subdirs in [("test", ["Test"])]:
            out_hhw = WORK_DIR / f"HHW_{mode}_{subset_name}"
            (out_hhw / "images" / "val").mkdir(parents=True, exist_ok=True)
            (out_hhw / "labels" / "val").mkdir(parents=True, exist_ok=True)
            n_img, n_box = 0, 0
            for sd in subdirs:
                ad = hhw_root / sd / "Annotation"
                jd = hhw_root / sd / "JPEGImage"
                if not ad.exists():
                    continue
                for xp in sorted(ad.glob("*.xml")):
                    stem = xp.stem
                    img_p = None
                    for ext in (".jpg", ".JPG", ".png", ".jpeg"):
                        c = jd / f"{stem}{ext}"
                        if c.exists():
                            img_p = c
                            break
                    if img_p is None:
                        continue
                    root = ET.parse(xp).getroot()
                    sz = root.find("size")
                    iw, ih = int(sz.find("width").text), int(sz.find("height").text)
                    lines = []
                    for obj in root.findall("object"):
                        cname = obj.find("name").text.strip().lower()
                        if cname not in cmap:
                            continue
                        cid = cmap[cname]
                        bb = obj.find("bndbox")
                        xmin = max(0.0, min(float(bb.find("xmin").text), float(iw)))
                        ymin = max(0.0, min(float(bb.find("ymin").text), float(ih)))
                        xmax = max(0.0, min(float(bb.find("xmax").text), float(iw)))
                        ymax = max(0.0, min(float(bb.find("ymax").text), float(ih)))
                        if xmax <= xmin or ymax <= ymin:
                            continue
                        cx = ((xmin + xmax) / 2.0) / iw
                        cy = ((ymin + ymax) / 2.0) / ih
                        bw = (xmax - xmin) / iw
                        bh = (ymax - ymin) / ih
                        lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                        n_box += 1
                    dst_name = f"{sd}_{img_p.name}"
                    dst_img = out_hhw / "images" / "val" / dst_name
                    if not dst_img.exists():
                        try:
                            os.symlink(img_p, dst_img)
                        except Exception:
                            shutil.copy2(img_p, dst_img)
                    (out_hhw / "labels" / "val" / f"{sd}_{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
                    n_img += 1
            yp = out_hhw / "hhw.yaml"
            yp.write_text(
                f"path: {out_hhw.resolve()}\ntrain: images/val\nval: images/val\nnc: {len(names_list)}\nnames: {names_list}\n",
                encoding="utf-8",
            )
            hhw_configs[f"{mode}_{subset_name}"] = yp
            print(f"  HHW {mode}_{subset_name}: {n_img} imgs, {n_box} boxes", flush=True)

# ==============================================================================
# 3. CREATE CLEAN COCO-PRETRAINED NC=2 INIT WEIGHT
# ==============================================================================
from ultralytics.nn.tasks import DetectionModel
base_coco = YOLO("yolo11s.pt")
nc2_model = DetectionModel("yolo11s.yaml", nc=2, verbose=False)
sd_src = base_coco.model.state_dict()
sd_dst = nc2_model.state_dict()
matched = {k: v for k, v in sd_src.items() if k in sd_dst and sd_dst[k].shape == v.shape}
sd_dst.update(matched)
nc2_model.load_state_dict(sd_dst)
nc2_model.names = {0: "hat", 1: "person"}
INIT_WEIGHT_PATH = WORK_DIR / "yolo11s_shwd_nc2_init.pt"
torch.save({"model": nc2_model}, INIT_WEIGHT_PATH)
os.environ["CURRENT_BASE_WEIGHT"] = str(INIT_WEIGHT_PATH)
print(f"[INIT] Saved clean COCO->nc2 weight ({len(matched)}/{len(sd_dst)} tensors) to {INIT_WEIGHT_PATH}", flush=True)

# Sanity check epoch-0 warm-start equivalence and fusion equivalence
m_a0 = build_ablation_model("A0", str(INIT_WEIGHT_PATH)).model.eval()
m_a3 = build_ablation_model("A3", str(INIT_WEIGHT_PATH)).model.eval()
dummy = torch.randn(1, 3, 640, 640)
with torch.no_grad():
    y0 = m_a0(dummy)[0]
    y3 = m_a3(dummy)[0]
    eq_diff = (y0 - y3).abs().max().item()
    for mod in m_a3.modules():
        if isinstance(mod, RepConv):
            mod.switch_to_deploy()
    y3_dep = m_a3(dummy)[0]
    fuse_diff = (y3 - y3_dep).abs().max().item()
print(f"[CHECK] Epoch-0 warm-start diff={eq_diff:.6f} | RepConv deploy fusion diff={fuse_diff:.6f}", flush=True)
del m_a0, m_a3

# ==============================================================================
# 4. RUN SEED 2026 CONTROLLED ABLATIONS (A0 -> A4)
# ==============================================================================
SEED = 2026
EPOCHS = 60
BATCH_SIZE = 32
IMGSZ = 640
PATIENCE = 20

ABLATIONS = [
    {"id": "A0", "name": "Baseline YOLO11s (CIoU)"},
    {"id": "A1", "name": "+ CoordConv Stem"},
    {"id": "A2", "name": "+ Neck RepConv"},
    {"id": "A3", "name": "+ BiFormer P4/P5"},
    {"id": "A4", "name": "+ Focal-EIoU (Proposed)"},
]

OUT_DIR = WORK_DIR / "SHWD V1 Benchmark_Ablation_Seed2026"
CKPT_DIR = OUT_DIR / "checkpoints"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

records = []
hhw_records = []
lat_records = []

def extract_metrics(val_res):
    res = {
        "mAP50": round(float(val_res.box.map50) * 100.0, 2),
        "mAP50_95": round(float(val_res.box.map) * 100.0, 2),
        "precision": round(float(val_res.box.mp) * 100.0, 2),
        "recall": round(float(val_res.box.mr) * 100.0, 2),
    }
    try:
        p0, r0, ap50_0, ap_0 = val_res.box.class_result(0)
        res.update({
            "hat_P": round(float(p0) * 100.0, 2),
            "hat_R": round(float(r0) * 100.0, 2),
            "hat_AP50": round(float(ap50_0) * 100.0, 2),
            "hat_AP50_95": round(float(ap_0) * 100.0, 2),
        })
    except Exception:
        pass
    try:
        p1, r1, ap50_1, ap_1 = val_res.box.class_result(1)
        res.update({
            "person_P": round(float(p1) * 100.0, 2),
            "person_R": round(float(r1) * 100.0, 2),
            "person_AP50": round(float(ap50_1) * 100.0, 2),
            "person_AP50_95": round(float(ap_1) * 100.0, 2),
        })
    except Exception:
        pass
    return res

def benchmark_latency_t4(pt_path: Path, ab_id: str):
    if not torch.cuda.is_available():
        return {}
    y = YOLO(str(pt_path))
    m = y.model.cuda().eval()
    for mod in m.modules():
        if isinstance(mod, RepConv):
            mod.switch_to_deploy()
    x32 = torch.randn(1, 3, 640, 640, device="cuda", dtype=torch.float32)
    with torch.no_grad():
        for _ in range(50):
            _ = m(x32)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(200):
            _ = m(x32)
        torch.cuda.synchronize()
        lat_fp32 = (time.perf_counter() - t0) / 200.0 * 1000.0
    m.half()
    x16 = x32.half()
    with torch.no_grad():
        for _ in range(50):
            _ = m(x16)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(200):
            _ = m(x16)
        torch.cuda.synchronize()
        lat_fp16 = (time.perf_counter() - t0) / 200.0 * 1000.0
    del m, y
    torch.cuda.empty_cache()
    return {
        "ablation_id": ab_id,
        "pyt_fp32_ms": round(lat_fp32, 2),
        "pyt_fp32_fps": round(1000.0 / lat_fp32, 1),
        "pyt_fp16_ms": round(lat_fp16, 2),
        "pyt_fp16_fps": round(1000.0 / lat_fp16, 1),
    }

for ab in ABLATIONS:
    ab_id = ab["id"]
    print(f"\n{'='*80}\n[TRAIN] {ab_id}: {ab['name']} | Seed={SEED} | Epochs={EPOCHS}\n{'='*80}", flush=True)
    os.environ["CURRENT_ABLATION_ID"] = ab_id
    model = build_ablation_model(ab_id, str(INIT_WEIGHT_PATH))
    train_params = sum(p.numel() for p in model.model.parameters())

    t_start = time.time()
    model.train(
        trainer=ControlledAblationTrainer,
        data=str(shwd_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        device=TRAIN_DEVICE,
        seed=SEED,
        deterministic=True,
        optimizer="SGD",
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        cos_lr=True,
        patience=PATIENCE,
        close_mosaic=10,
        cache=True,
        save=True,
        val=True,
        plots=False,
        project=str(OUT_DIR / "runs"),
        name=f"{ab_id}_seed{SEED}",
        exist_ok=True,
        verbose=False,
    )
    train_min = (time.time() - t_start) / 60.0

    best_pt = Path(model.trainer.save_dir) / "weights" / "best.pt"
    deploy_pt = CKPT_DIR / f"{ab_id}_seed{SEED}_deploy.pt"

    # Load best.pt, fuse RepConv, verify deploy equivalence, save clean deploy checkpoint
    eval_yolo = YOLO(str(best_pt))
    for mod in eval_yolo.model.modules():
        if isinstance(mod, RepConv):
            mod.switch_to_deploy()
    deploy_params = sum(p.numel() for p in eval_yolo.model.parameters())
    eval_yolo.save(str(deploy_pt))

    # Strict held-out test set evaluation (1,517 images, imgsz=640, no TTA)
    test_res = eval_yolo.val(
        data=str(shwd_yaml),
        split="test",
        imgsz=IMGSZ,
        batch=16,
        device=EVAL_DEVICE,
        half=False,
        augment=False,
        verbose=False,
    )
    m_dict = extract_metrics(test_res)
    rec = {
        "ablation_id": ab_id,
        "name": ab["name"],
        "seed": SEED,
        "train_params": train_params,
        "deploy_params": deploy_params,
        "train_time_min": round(train_min, 2),
        **m_dict,
    }
    records.append(rec)
    pd.DataFrame(records).to_csv(OUT_DIR / "shwd_test_ablation_seed2026.csv", index=False)
    print(f"[TEST {ab_id}] {rec}", flush=True)

    # Latency measurement on T4
    lat_rec = benchmark_latency_t4(deploy_pt, ab_id)
    lat_records.append(lat_rec)
    pd.DataFrame(lat_records).to_csv(OUT_DIR / "t4_latency_seed2026.csv", index=False)

    # Cross-Domain Zero-Shot on HHW for A0, A3, A4
    if ab_id in ("A0", "A3", "A4") and hhw_configs:
        for hname, hyp in hhw_configs.items():
            # For hatonly yaml (nc=1), evaluate hat class (cls 0)
            if "hatonly" in hname:
                hr = eval_yolo.val(
                    data=str(hyp), split="val", imgsz=IMGSZ, batch=16,
                    device=EVAL_DEVICE, half=False, augment=False, classes=[0], verbose=False,
                )
            else:
                hr = eval_yolo.val(
                    data=str(hyp), split="val", imgsz=IMGSZ, batch=16,
                    device=EVAL_DEVICE, half=False, augment=False, verbose=False,
                )
            hm = extract_metrics(hr)
            hrec = {"ablation_id": ab_id, "hhw_eval": hname, **hm}
            hhw_records.append(hrec)
            pd.DataFrame(hhw_records).to_csv(OUT_DIR / "hhw_zeroshot_seed2026.csv", index=False)
            print(f"[HHW {ab_id} | {hname}] {hm}", flush=True)

    del model, eval_yolo
    torch.cuda.empty_cache()
    gc.collect()

print("\n[SUMMARY] SHWD Held-Out Test (Seed 2026):", flush=True)
print(pd.DataFrame(records).to_string(index=False), flush=True)
if hhw_records:
    print("\n[SUMMARY] HHW Zero-Shot:", flush=True)
    print(pd.DataFrame(hhw_records).to_string(index=False), flush=True)
print("\n[SUMMARY] Tesla T4 Latency:", flush=True)
print(pd.DataFrame(lat_records).to_string(index=False), flush=True)
